<h1>4주차 복습</h1>

1. knn(거리기반 알고리즘)을 이용하여 분류를 시도하시오
, 단 스케일링과 k-fold를 꼭 시도아시오.


2. K_means를 이용하여 데이터를 분석하시오.
단, 실루엣 계수를 이용하여 최적의 K값을 찾으시오

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd


X, _ = load_iris(return_X_y=True)


X_scaled = StandardScaler().fit_transform(X)

best_k, best_score = None, -1
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f"K={k}: 실루엣 계수 = {score:.4f}")
    if score > best_score:
        best_k, best_score = k, score

print(f"\n[최적 K] {best_k}  (실루엣 계수 = {best_score:.4f})")

model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = model.fit_predict(X_scaled)

print("[군집] 할당된 클러스터 번호:", cluster_labels[:10])

K=2: 실루엣 계수 = 0.5818
K=3: 실루엣 계수 = 0.4599
K=4: 실루엣 계수 = 0.3869
K=5: 실루엣 계수 = 0.3459
K=6: 실루엣 계수 = 0.3171
K=7: 실루엣 계수 = 0.3202
K=8: 실루엣 계수 = 0.3387
K=9: 실루엣 계수 = 0.3424
K=10: 실루엣 계수 = 0.3518

[최적 K] 2  (실루엣 계수 = 0.5818)
[군집] 할당된 클러스터 번호: [1 1 1 1 1 1 1 1 1 1]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import numpy as np

X = np.array([[1, 2], [1, 4], [1, 0],
              [10, 2], [10, 4], [10, 0]])

kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(X)


average_score = silhouette_score(X, labels)

sample_scores = silhouette_samples(X, labels)

print(f"예측된 군집 레이블: {labels}")
print(f"전체 평균 실루엣 점수: {average_score:.4f}\n")

print("--- 데이터 포인트별 개별 실루엣 계수 ---")
for i, score in enumerate(sample_scores):
    print(f"데이터 {X[i]} -> 군집 {labels[i]} | 실루엣 계수: {score:.4f}")


예측된 군집 레이블: [0 0 0 1 1 1]
전체 평균 실루엣 점수: 0.7133

--- 데이터 포인트별 개별 실루엣 계수 ---
데이터 [1 2] -> 군집 0 | 실루엣 계수: 0.7813
데이터 [1 4] -> 군집 0 | 실루엣 계수: 0.6794
데이터 [1 0] -> 군집 0 | 실루엣 계수: 0.6794
데이터 [10  2] -> 군집 1 | 실루엣 계수: 0.7813
데이터 [10  4] -> 군집 1 | 실루엣 계수: 0.6794
데이터 [10  0] -> 군집 1 | 실루엣 계수: 0.6794


# 5주차 실습 · PyTorch 시작 — 텐서와 학습 루프
**피지컬AI응용 (MR347) · 2026-2 · 인하공업전문대학 컴퓨터시스템공학과**

강의 슬라이드의 실습 ①~④ 입니다. 데이터는 같은 **배터리·속도 → 주행거리** 회귀 데이터입니다.
오늘의 목표는 `LinearRegression().fit()` 으로 얻은 **계수 `[12.09, -142.55]` 를 PyTorch 학습 루프로 똑같이 얻어내는 것**입니다.

| 실습 | 내용 | 핵심 |
|---|---|---|
| ① | 텐서 만들기 · shape · dtype · numpy 왕복 | `torch.tensor(..., dtype=torch.float32)` |
| ② | 자동미분 · 경사하강 한 줄 | `requires_grad` → `backward()` → `.grad` |
| ③ | 3주차 회귀를 PyTorch 로 — 다섯 줄 루프 | `zero_grad → model → loss → backward → step` |
| ③-2 | 표준화를 빼면 무슨 일이 생기나 | 신경망이 죽는 1번 원인 |
| ④ | 층 하나 더 얹기 — MLP | `nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1)` |



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("torch", torch.__version__)
print("준비 완료")


torch 2.11.0+cpu
준비 완료


In [ ]:
# ===== 데이터 준비 (그대로 실행) — 회귀 데이터와 완전히 같음 =====
rng = np.random.default_rng(42)
n = 120
battery = rng.uniform(20, 100, n)
speed   = rng.uniform(0.5, 2.0, n)
distance = 12 * battery - 150 * speed + rng.normal(0, 20, n)   # 우리가 진짜
reg = pd.DataFrame({"배터리": battery.round(1), "속도": speed.round(2), "주행거리": distance.round(1)})

X = reg[["배터리", "속도"]].values
y = reg["주행거리"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   # 1 나누고

# 답 (오늘 이 숫자를 PyTorch 로 똑같이 얻어내는 것이 목표)
lr_sk = LinearRegression().fit(X_train, y_train)
print("3주차 sklearn 계수:", lr_sk.coef_.round(2), "절편:", round(lr_sk.intercept_, 2))
print("3주차 sklearn 시험 MSE:", round(mean_squared_error(y_test, lr_sk.predict(X_test)), 2),
      "R2:", round(r2_score(y_test, lr_sk.predict(X_test)), 4))
print("X_train:", X_train.shape, "y_train:", y_train.shape)


3주차 sklearn 계수: [  12.09 -142.55] 절편: -15.74
3주차 sklearn 시험 MSE: 513.41 R2: 0.9925
X_train: (96, 2) y_train: (96,)


**실행 결과** (torch 버전 줄은 달라도 됩니다)
```
3주차 sklearn 계수: [  12.09 -142.55] 절편: -15.74
3주차 sklearn 시험 MSE: 513.41 R2: 0.9925
X_train: (96, 2) y_train: (96,)
```

---
### 실습 ① 텐서 다루기

다음 네 가지를 순서대로 출력하세요.

1. `torch.arange(6)` 을 `(2, 3)` 으로 reshape 한 텐서와 그 `shape`
2. `X_train`, `y_train` 을 **float32 텐서** `X_t`, `y_t` 로 만들기 — `y_t` 는 `(96, 1)` 모양. 두 텐서의 `shape` 와 `dtype` 출력
3. `X_t` 의 열별 평균 `X_t.mean(dim=0)` (numpy 의 `axis=0` 과 같은 뜻)
4. `y_t` 를 다시 numpy 로 되돌려 앞 3개 출력

> 💡 **힌트** — `torch.tensor(X_train, dtype=torch.float32)` / `.reshape(-1, 1)` / `.mean(dim=0)` / `.numpy()[:3]`

**예상 출력**
```
tensor([[0, 1, 2],
        [3, 4, 5]])
torch.Size([2, 3])
torch.Size([96, 2]) torch.float32
torch.Size([96, 1]) torch.float32
tensor([58.8625,  1.2299])
[[770.2]
 [630.4]
 [354.6]]
```


In [ ]:
# 여기에 코드를 쓰세요
import torch
import numpy as np

a = torch.arange(6).reshape(2,3)
print(a)
print(a.shape)
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
print(X_t.shape, X_t.dtype)
print(y_t.shape, y_t.dtype)
print(X_t.mean(dim=0))
print(y_t.numpy()[:3])


tensor([[0, 1, 2],
        [3, 4, 5]])
torch.Size([2, 3])
torch.Size([96, 2]) torch.float32
torch.Size([96, 1]) torch.float32
tensor([58.8625,  1.2299])
[[770.2]
 [630.4]
 [354.6]]


---
### 실습 ② 자동미분 — 손으로 기울기

1. `x = 2.0` (requires_grad=True) 에서 `y = 3 * x**2` 의 기울기를 `backward()` 로 구해 `x.grad` 출력 (답: 12)
2. `w = [1., 2., 3.]` 에 대해 `z = (w**2).sum()` 의 기울기 출력 (답: [2, 4, 6])
3. 2번의 `z` 를 다시 만들어 `backward()` 를 **한 번 더** 호출한 뒤 `w.grad` 출력 — 어떻게 되나?
4. 표준화한 3주차 데이터로 `w -= lr * w.grad` 경사하강을 **30번** 돌리며 손실을 5번마다 출력 (아래 준비 코드 사용)

```python
# 4번 준비 — 표준화 (X 도 y 도)
sx = StandardScaler().fit(X_train);  sy = StandardScaler().fit(y_train.reshape(-1, 1))
X_s = torch.tensor(sx.transform(X_train), dtype=torch.float32)
y_s = torch.tensor(sy.transform(y_train.reshape(-1, 1)), dtype=torch.float32)
w = torch.zeros(2, 1, requires_grad=True);  b = torch.zeros(1, requires_grad=True);  lr = 0.1
```

> 💡 **힌트** — `pred = X_s @ w + b` → `loss = ((pred - y_s)**2).mean()` → `loss.backward()` → `with torch.no_grad(): w -= lr * w.grad; b -= lr * b.grad` → `w.grad.zero_(); b.grad.zero_()`

**예상 출력**
```
1) tensor(12.)
2) tensor([2., 4., 6.])
3) tensor([ 4.,  8., 12.])
4) 0 1.0
4) 5 0.1179
4) 10 0.0185
4) 15 0.0072
4) 20 0.0059
4) 25 0.0057
w: [ 0.977 -0.24 ]
```


In [ ]:
# 여기에 코드를 쓰세요
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x**2
y.backward()
print("1)", x.grad)
w = torch.tensor([1., 2., 3.], requires_grad=True)
z = (w**2).sum()
z.backward()
print("2)",w.grad)


sx = StandardScaler().fit(X_train);  sy = StandardScaler().fit(y_train.reshape(-1, 1))
X_s = torch.tensor(sx.transform(X_train), dtype=torch.float32)
y_s = torch.tensor(sy.transform(y_train.reshape(-1, 1)), dtype=torch.float32)
w = torch.zeros(2, 1, requires_grad=True);  b = torch.zeros(1, requires_grad=True);  lr = 0.1

for i in range(30):
  pred = X_s @ w + b
  loss = ((pred - y_s)**2).mean()
  loss.backward()
  with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
  w.grad.zero_(); b.grad.zero_()
  if i % 5 == 0:
    print(f"{i} {loss.item():.4f}")
print("w:", w.detach().numpy().flatten())


1) tensor(12.)
2) tensor([2., 4., 6.])
0 1.0000
5 0.1179
10 0.0185
15 0.0072
20 0.0059
25 0.0057
w: [ 0.9765604 -0.2401332]


---
### 실습 ③ 회귀를 PyTorch 로

3주차 5단계 골격 그대로, '학습하고' 칸만 다섯 줄로 펼칩니다. 실습 ②의 `X_s`, `y_s`, `sx`, `sy` 를 그대로 씁니다.

1. `torch.manual_seed(0)` 후 부품 세 개 — `model = nn.Sequential(nn.Linear(2, 1))`, `loss_fn = nn.MSELoss()`, `opt = optim.SGD(model.parameters(), lr=0.1)`
2. **다섯 줄 루프** 100 epoch, 10번마다 손실 출력
3. 학습된 가중치를 원래 단위로 되돌려 계수 출력 - `w * sy.scale_ / sx.scale_`
4. 시험 데이터로 예측(표준화 → 모델 → 역변환)해 **MSE 와 R²** 출력 → 이전 숫자와 비교
5. `[[60, 1.2]]` 의 예측값

> 💡 **힌트** — 루프: `opt.zero_grad(); pred = model(X_s); loss = loss_fn(pred, y_s); loss.backward(); opt.step()` / 가중치: `model[0].weight.detach().numpy().flatten()` / 예측: `with torch.no_grad(): sy.inverse_transform(model(torch.tensor(sx.transform(X_test), dtype=torch.float32)).numpy())`

**예상 출력**
```
0 1.6438
10 0.0278
20 0.006
30 0.0057
40 0.0057
50 0.0057
60 0.0057
70 0.0057
80 0.0057
90 0.0057
계수 (원래 단위): [  12.09 -142.55]
시험 MSE: 513.41 R2: 0.9925
predict [[60, 1.2]] → [[538.8]]
```


In [67]:

torch.manual_seed(0)   # 오타 수정: maual_seed -> manual_seed
model = nn.Sequential(nn.Linear(2, 1))
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    opt.zero_grad()
    pred = model(X_t)
    loss = loss_fn(pred, y_t)
    loss.backward()
    opt.step()
    if epoch % 10 == 0:
        print(epoch, round(loss.item(),4))
w = model[0].weight.detach().numpy().flatten()
print("계수 (원래 단위):", w * sy.scale_ / sx.scale_)


with torch.no_grad():
    pred_test = sy.inverse_transform(
        model(torch.tensor(sx.transform(X_test), dtype=torch.float32)).numpy()
    )
    print('MSE', mean_squared_error(y_test, pred_test), 'R2', r2_score(y_test, pred_test))

0 1.6438
10 0.0278
20 0.006
30 0.0057
40 0.0057
50 0.0057
60 0.0057
70 0.0057
80 0.0057
90 0.0057
계수 (원래 단위): [  12.09378951 -142.54993459]
MSE 513.4108926439084 R2 0.9925416649207021


---
### 실습 ③-2 표준화를 빼면 - 신경망이 죽는 가장 흔한 방식

표준화하지 **않은** `X_train`, `y_train` 을 그대로 float32 텐서로 만들어 실습 ③과 같은 루프를 **5 epoch** 만 돌리고 손실을 매번 출력하세요 (`lr=0.01`).

마지막에 한 문장으로 답하세요: *손실이 어떻게 되었고, 왜 그런가?*

> 💡 **힌트** — `X_raw = torch.tensor(X_train, dtype=torch.float32)`, `y_raw = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)` → 같은 다섯 줄

**예상 출력**
```
0 343558.59375
1 2023563648.0
2 12225763868672.0
3 7.386471011765453e+16
4 4.462699554035833e+20
```


In [ ]:
# 여기에 코드를 쓰세요


---
### 실습 ④ 층 하나 더 얹기 - MLP

모델만 바꿔 실습 ③을 반복합니다. 다섯 줄은 한 글자도 바꾸지 마세요.

- `model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))`, `opt = optim.Adam(model.parameters(), lr=0.05)`, **300 epoch**
- 파라미터 개수(`sum(p.numel() for p in model.parameters())`)와 시험 **R²** 를 출력하고, 실습 ③(파라미터 3개, R² 0.9925)과 비교하세요.

> 💡 **힌트** — 루프는 실습 ③ 그대로. 예측·역변환도 그대로.

**예상 출력** (실습 ④ 의 R² 는 소수 셋째 자리에서 조금 달라도 됩니다)
```
파라미터 개수: 65
MLP 시험 R2: 0.9912  (선형층 하나: 0.9925)
```


In [ ]:
# 여기에 코드를 쓰세요


---
## 과제 3 (4~5주) - Keras 코드를 PyTorch 다섯 줄로

지난 학기 Keras 로 짰던 코드 중 **회귀 또는 분류 하나**를 골라 아래 형식으로 옮기세요. 데이터는  자유.

```python
# (1) 데이터 → float32 텐서 (표준화 포함, y 는 (n,1))
# (2) 부품 세 개: model / loss_fn / opt
# (3) 다섯 줄 루프 (epoch 수와 lr 은 직접 정하고, 손실을 10번마다 출력)
# (4) 시험 데이터 점수 (MSE·R² 또는 정확도)
# (5) 한 문장: Keras 의 fit() 과 비교해 무엇이 같고 무엇이 달랐나
```

**제출** — 이 노트북(실습 ①~④ 실행 결과 포함) + 과제 3 셀. 실습 ③-2 의 답 문장은 숫자와 함께.

**다음 시간 시작 10분** — 암기 카드 ①~⑤ (부품 세 개 + 다섯 줄 + no_grad 예측) 빈 화면 재현.


In [ ]:
# 과제 3 — 여기에 Keras → PyTorch 코드를 쓰세요
